# lncRNA-DNAtriplex Example Usage

Links to open-source datasets:

[cBioPortal](https://www.cbioportal.org/datasets) \
[RefSeq](https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.40_GRCh38.p14/) \
[LNCipedia](https://lncipedia.org/download)

## Dataset breakdown

### cBioPortal
data_mutations.txt — cBioPortal somatic mutation data, MAF format. Each row is one mutation event in one patient: Hugo symbol, chromosome, start/end position, consequence, and variant classification. This is the source for mutated_dict.

This example uses the mutation data for uterine endometrial carcinoma (MSK 2024 cohort, ucec_msk_2024), a random example pulled from cBioPortal. From the cohort folder, data_mutations.txt is used.

In [8]:
%%bash

!wget https://datahub.assets.cbioportal.org/ucec_msk_2024.tar.gz
!tar -xvzf ucec_msk_2024.tar.gz

--2026-03-25 22:08:29--  https://datahub.assets.cbioportal.org/ucec_msk_2024.tar.gz
Resolving datahub.assets.cbioportal.org (datahub.assets.cbioportal.org)... 108.138.64.115, 108.138.64.113, 108.138.64.70, ...
Connecting to datahub.assets.cbioportal.org (datahub.assets.cbioportal.org)|108.138.64.115|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126898 (124K) [application/x-tar]
Saving to: ‘ucec_msk_2024.tar.gz’

ucec_msk_2024.tar.g 100%[===================>] 123.92K  --.-KB/s    in 0.02s   

2026-03-25 22:08:29 (6.91 MB/s) - ‘ucec_msk_2024.tar.gz’ saved [126898/126898]



### RefSeq
Healthy reference genome. Three necessary files used
1. p14_genomic.fna.gz — The full human reference genome (GRCh38.p14) in FASTA format. One sequence record per chromosome/contig, identified by RefSeq accession IDs like NC_000001.11. This is what pyfaidx indexes to pull gene body sequences by coordinate.
2. p14_genomic.gtf.gz — Gene annotation for GRCh38.p14. Tab-delimited, one row per genomic feature (gene, transcript, exon, CDS, etc.). Gives you the chromosomal coordinates and strand for every gene by Hugo symbol — this is what _parse_gtf_for_genes() reads to know where to slice the FASTA.
3. p14_genomic.assembly_report.txt — A mapping table that links human-readable chromosome names (1, chrX, MT) to their RefSeq accession IDs (NC_000001.11). Needed because the GTF uses one naming convention and the FASTA uses the other — load_chrom_map() bridges them.

In [ ]:
%%bash

!wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.40_GRCh38.p14/GCF_000001405.40_GRCh38.p14_genomic.gtf.gz
!wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.40_GRCh38.p14/GCF_000001405.40_GRCh38.p14_genomic.fna.gz
!wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.40_GRCh38.p14/GCF_000001405.40_GRCh38.p14_assembly_report.txt

### LNCipedia
lncipedia_5_2_hc.fasta — LNCipedia v5.2 high-confidence lncRNA sequences in FASTA format. The hc designation means these transcripts passed LNCipedia's quality filters (non-coding probability, multi-source validation). Each header is a transcript ID like lnc-TP53-1:1. This populates lncrna_dict.

In [ ]:
!wget https://lncipedia.org/downloads/lncipedia_5_2/full-database/lncipedia_5_2.fasta

## File Previews

In [ ]:
def preview(f, nlines = 10):
    with open(f, 'r') as f:
        for i, line in enumerate(f, 1):
            if i > nlines:
                break
            print(line.strip())

In [ ]:
main = Path.cwd()
# RefSeq
genomeF = main / "GCF_000001405.40_GRCh38.p14_genomic.fna.gz"
annotationF = main / "GCF_000001405.40_GRCh38.p14_genomic.gtf.gz"
assemblyF = main / "GCF_000001405.40_GRCh38.p14_assembly_report.txt"

# cBioPortal
mutationsF = [main / "ucec_msk_2024" / "data_mutations.txt"]

# LNCipedia
lncrnaF = main / "lncipedia_5_2_hc.fasta"

In [ ]:
df = pd.read_csv(mutationsF, sep = "\t")
df.head(50)

In [ ]:
preview(assemblyF)

## Pipeline

In [15]:
import pandas as pd
from Bio import SeqIO
from pyfaidx import Fasta
from pathlib import Path
import argparse
import subprocess
import os
from datetime import datetime
from typing import Dict, List

In [2]:
!pwd

/net/dali/home/mscbio/rop174/02604/lncRNA-DNAtriplex


In [3]:
import sys
sys.path.insert(0, "./src")  # relative to where the notebook lives

from rules       import transfer_string, reverse_seq, complement
from stats       import calc_score
from sim         import SIM, Triplex, cluster_triplex, print_cluster
from data_loader import load_mutations, load_healthy_sequences, load_lncrnas, load_chrom_map
from alignment import Para, long_target, report

In [4]:
args = argparse.Namespace(
    mutations = mutationsF,
    genome = genomeF,
    annotation = annotationF,
    lncrna = lncrnaF,
    assembly = assemblyF,
    outpath = "./results")

para = Para()

In [5]:
# ── ensure output directory exists ──────────────────────────────────
os.makedirs(args.outpath, exist_ok=True)
print(f"Output directory: {args.outpath}")

Output directory: ./results


In [17]:
# ── load dictionaries ────────────────────────────────────────────────
print("Loading mutated gene data ...")
mutated_dict, gene_names = load_mutations(args.mutations)

print("Building chromosome map ...")
chrom_map = load_chrom_map(args.assembly)

print("Loading healthy reference sequences ...")
healthy_dict = load_healthy_sequences(
    fasta_path      = args.genome,
    gtf_path        = args.annotation,
    gene_names      = gene_names,
    chrom_map       = chrom_map
)

print("Loading lncRNA sequences ...")
lncrna_dict = load_lncrnas(args.lncrna)

print(f"\n=== Data Summary ===")
print(f"  Mutated genes:    {len(mutated_dict)}")
print(f"  Healthy seqs:     {len(healthy_dict)}")
print(f"  lncRNA entries:   {len(lncrna_dict)}")

Loading mutated gene data ...
[INFO] Loaded 381 unique mutated genes from 1 file(s).
Building chromosome map ...
[INFO] Loaded 705 chromosome mappings from GCF_000001405.40_GRCh38.p14_assembly_report.txt.
[INFO] Total chromosome mappings after merge: 1410
Loading healthy reference sequences ...
[INFO] FASTA is already BGZF compressed.
[INFO] Indexing genome FASTA (this may take a moment the first time)…
[INFO] Parsing GTF annotation…


[WARNING] 18 genes had no GTF entry: ['FAM175A', 'CDKN2Ap14ARF', 'STK19', 'WHSC1L1', 'HIST1H3B', 'HIST1H3C', 'MLL3', 'HIST1H3E', 'PARK2', 'FAM58A']...


[INFO] Retrieved reference sequences for 363 genes.
Loading lncRNA sequences ...
[INFO] Loaded 107039 lncRNA transcripts.

=== Data Summary ===
  Mutated genes:    381
  Healthy seqs:     363
  lncRNA entries:   107039


In [19]:
# ── prepare gene lists (genes present in BOTH dicts) ─────────────────
shared_genes   = sorted(g for g in gene_names if g in healthy_dict)
lncrna_ids     = list(lncrna_dict.keys())

test_n         = 1
test_lncrnas   = lncrna_ids[:test_n]

print(f"\n  Genes available for analysis: {len(shared_genes)}")
print(f"  lncRNAs to test:              {len(test_lncrnas)}")
print()


  Genes available for analysis: 363
  lncRNAs to test:              1



In [ ]:
# ── main loop ────────────────────────────────────────────────────────
# Outer: first {test_n} lncRNAs
# Inner: all genes found in both healthy_dict and mutated_dict
# Each (lncRNA, gene) pair is run against both the healthy and mutated seq

loop_start = datetime.now()
print(f"Loop started at: {loop_start.strftime('%Y-%m-%d %H:%M:%S')}\n")

# results : {lnc_id: {gene: {"healthy": [Triplex], mut_label: [Triplex], ...}}}
results: Dict[str, Dict[str, Dict[str, List[Triplex]]]] = {}
total_pairs = 0
total_hits  = 0

for lnc_idx, lnc_id in enumerate(test_lncrnas):
    lnc_seq = lncrna_dict[lnc_id]
    print(f"[{lnc_idx + 1}/{len(test_lncrnas)}] lncRNA: {lnc_id} "
          f"({len(lnc_seq)} nt)")
    results[lnc_id] = {}

    for gene in shared_genes:
        lnc_seq     = lncrna_dict[lnc_id]
        healthy_seq = healthy_dict[gene]
        gene_info   = mutated_dict[gene]
        results[lnc_id][gene] = {}

        # ── healthy run ──────────────────────────────────────────────
        healthy_triplexes = long_target(para, lnc_seq, healthy_seq)
        results[lnc_id][gene]["healthy"] = healthy_triplexes
        total_hits += len(healthy_triplexes)

        # ── mutated runs ─────────────────────────────────────────────
        for mut_idx, mut in enumerate(gene_info.get("mutations", [])):
            mut_label = f"mut{mut_idx}_{mut.get('variant_class', 'unknown')}"
            mutated_triplexes = long_target(para, lnc_seq, healthy_seq)
            results[lnc_id][gene][mut_label] = mutated_triplexes
            total_hits += len(mutated_triplexes)

        total_pairs += 1

loop_end = datetime.now()
elapsed  = loop_end - loop_start

print(f"\nLoop ended at:   {loop_end.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Elapsed time:    {elapsed}")
print(f"Total (lncRNA × gene) pairs processed: {total_pairs}")
print(f"Total triplex hits across all runs:     {total_hits}")

# ── write report ─────────────────────────────────────────────────────
report(results, mutated_dict, args.outpath)

Loop started at: 2026-03-26 14:52:46

[1/1] lncRNA: LINC01725:44 (1991 nt)
  DNA chunk @ 0 (5000 bp) ...


In [7]:
# Debugging

# Check FASTA sequence IDs
from pyfaidx import Fasta
genome = Fasta(args.genome)
print(list(genome.keys())[:5])

# Check GTF chromosome names and gene attributes
import gzip
with gzip.open(args.annotation, "rt") as f:
    for i, line in enumerate(f):
        if not line.startswith("#"):
            print(line[:200])
            break

# Check what attribute keys exist for a gene feature
import gzip
with gzip.open(args.annotation, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) > 2 and parts[2] == "gene":
            print(parts[8])
            break

# Check a few gene names from your mutation data
print(list(gene_names)[:10])

from data_loader import _parse_gtf_attributes
import gzip

gtf_genes = set()
with gzip.open(args.annotation, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) > 2 and parts[2] == "gene":
            attrs = _parse_gtf_attributes(parts[8])
            sym = attrs.get("gene_name") or attrs.get("gene") or attrs.get("gene_id", "")
            gtf_genes.add(sym)

print(f"Total genes in GTF: {len(gtf_genes)}")
print(f"Overlap with mutation genes: {len(gene_names & gtf_genes)}")
print(f"Sample GTF genes: {list(gtf_genes)[:10]}")
print(f"Missing from GTF: {list(gene_names - gtf_genes)[:10]}")

# Also check the raw attribute string to make sure parsing is correct
import gzip
with gzip.open(args.annotation, "rt") as f:
    count = 0
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) > 2 and parts[2] == "gene":
            print(repr(parts[8]))
            count += 1
            if count == 3:
                break

# What chroms does the GTF produce for your genes?
import gzip
from data_loader import _parse_gtf_attributes

sample_chroms = set()
with gzip.open(args.annotation, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) > 2 and parts[2] == "gene":
            attrs = _parse_gtf_attributes(parts[8])
            sym = attrs.get("gene_name") or attrs.get("gene") or attrs.get("gene_id", "")
            if sym in gene_names:
                sample_chroms.add(parts[0])

print("GTF chroms for your genes:", list(sample_chroms)[:10])

# What keys does the FASTA actually have?
from pyfaidx import Fasta
genome = Fasta(args.genome)
print("FASTA keys:", list(genome.keys())[:10])

# What does chrom_map produce for one of these?
print("chrom_map sample:", {k: chrom_map[k] for k in list(sample_chroms)[:5] if k in chrom_map})

NC_000001.11	BestRefSeq	gene	11874	14409	.	+	.	gene_id "DDX11L1"; transcript_id ""; db_xref "GeneID:100287102"; db_xref "HGNC:HGNC:37102"; description "DEAD/H-box helicase 11 like 1 (pseudogene)"; gbk
